In [ ]:
df_primary = combined_feed.trips.copy()
id_col = "trip_id"
primary_table = "trips"
identity_cols = ["service_id", "trip_headsign", "trip_short_name", "direction_id"]
foreign_keys = [("stop_times", "trip_id"), ]
df_primary

#df_primary = df_primary.sort_values(["trip_id"])
df_st = feed.stop_times.sort_values(["trip_id", "stop_sequence"])

df_st['_row_sig'] = pd.util.hash_pandas_object(
    df_st[["stop_id", "arrival_time", "departure_time"]],
    index=False
)

pattern = (
    df_st.groupby("trip_id")["_row_sig"]
    .apply(tuple)
    .map(hash)
    .rename("pattern_sig")
)
df_st = df_st.drop(columns=["_row_sig"])  # save memory
df_primary["pattern_sig"] = df_primary["trip_id"].map(pattern)

df_primary["trip_sig"] = pd.util.hash_pandas_object(
    df_primary[["route_id", "service_id", "direction_id", "shape_id", "pattern_sig"]],
    index=False
)

canonical = (
    df_primary
    .groupby("trip_sig")["trip_id"]
    .min()
)

trip_map = (
    df_primary.set_index("trip_id")["trip_sig"]
    .map(canonical)
)

df_primary["trip_id"] = df_primary["trip_id"].map(trip_map).fillna(df_primary["trip_id"])
df_st["trip_id"] = df_st["trip_id"].map(trip_map).fillna(df_st["trip_id"])

df_primary = df_primary.drop_duplicates("trip_id")

df_st = (
    df_st
    .sort_values(["trip_id", "stop_sequence"])
    .drop_duplicates(["trip_id", "stop_sequence"])
)
df_primary = (
    df_primary
    .sort_values(["trip_id"])
    .drop_duplicates(["trip_id"])
    .drop(columns=["pattern_sig", "trip_sig"], errors="ignore")
)

df_primary = df_primary.drop(columns=["pattern_sig", "trip_sig"], errors="ignore")

feed.trips = df_primary.reset_index(drop=True)
feed.stop_times = df_st.reset_index(drop=True)

final_count = len(df_primary)
duplicates_removed = initial_count - final_count

df_primary = combined_feed.shapes.copy()
id_col = "shape_id"
primary_table = "shapes"
identity_cols = ["shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"]
foreign_keys = [("trips", "shape_id"), ]
df_primary

use_identity_cols = identity_cols
df_primary.info()
sequence_cols = ["shape_pt_sequence"]

# For sequence tables: create signature from all rows grouped by ID
df_sorted = df_primary.sort_values([id_col] + sequence_cols)
df_sorted[id_col] = (
        df_sorted['feed_id'].astype(str)
        .str.replace("GTFS_", "", regex=False)
        .str.replace(".zip", "", regex=False)
        + "_"
        + df_sorted[id_col]
)
# Create row-level signature by concatenating identity columns
df_sorted['_row_sig'] = pd.util.hash_pandas_object(
    df_sorted[use_identity_cols],
    index=False
)
df_sorted
# Group and concatenate row signatures into single signature per ID
signatures = (
    df_sorted
    .groupby(id_col)["_row_sig"]
    .apply(tuple)
    .map(hash)
    .rename("_signature")
    .reset_index()
)
signatures
# Map signature to canonical (minimum) ID
canonical_map = (
    signatures
    .groupby('_signature')[id_col]
    .min()
)
canonical_map
# Create ID to canonical ID mapping
id_to_canonical = (
    signatures
    .set_index(id_col)['_signature']
    .map(canonical_map)
)  #For foreign key
id_to_canonical.index[df]
df_sorted[
    df_sorted[id_col].isin(id_to_canonical.index[id_to_canonical == "20250102_100"])
]
# Get set of canonical IDs
canonical_ids = canonical_map.values
canonical_ids
# Update primary table: keep only rows with canonical IDs
df_primary = (
    df_primary[df_primary[id_col].isin(canonical_ids)]
    .reset_index(drop=True)
    .drop_duplicates(subset=use_identity_cols + [id_col], keep="first")
)


In [ ]:
df_primary = combined_feed.agency.copy()
id_col = "agency_id"
primary_table = "agency"
identity_cols = ["agency_name", "agency_timezone"]
foreign_keys = [("routes", "agency_id")]
df_primary.loc[19, "agency_id"] = "999"
df_primary

In [ ]:
initial_count = len(df_primary)

# Filter identity_cols to those present in the dataframe
use_identity_cols = [c for c in identity_cols if c in df_primary.columns and c != id_col]
# For simple tables: group by identity columns directly
df_sorted = df_primary.sort_values(id_col)
df_sorted
# Map each unique combination of identity cols to canonical (minimum) ID
canonical_df = (
    df_sorted
    .groupby(use_identity_cols, dropna=False)[id_col]
    .min()
    .reset_index()
)
canonical_df
# Create mapping from all IDs to canonical IDs
id_to_canonical = (
    df_sorted[[id_col] + use_identity_cols]
    .merge(canonical_df, on=use_identity_cols, suffixes=('', '_canonical'))
    .drop_duplicates(subset=[id_col, f'{id_col}_canonical'])
    .set_index(id_col)[f'{id_col}_canonical']
)  #For foreign key
id_to_canonical
# Update primary table: keep only canonical rows
canonical_ids = canonical_df[id_col]
df_primary = (
    df_primary[df_primary[id_col].isin(canonical_ids)]
    .reset_index(drop=True)
    .drop_duplicates(subset=use_identity_cols + [id_col], keep="first")
)
df_primary
fk_df = combined_feed.routes.copy()
fk_df.loc[1, "agency_id"] = "999"
fk_df
# Map foreign keys to canonical IDs
fk_df[id_col] = fk_df[id_col].map(id_to_canonical).fillna(fk_df[id_col])
duplicates_removed = initial_count - final_count

In [ ]:
df_primary = combined_feed.stops.copy()
id_col = "stop_id"
primary_table = "stops"
identity_cols = ["stop_lat", "stop_lon", "stop_name", "stop_id"]
foreign_keys = [
    ("stop_times", "stop_id"),
    ("transfers", "from_stop_id"),
    ("transfers", "to_stop_id"),
]
df_primary
initial_count = len(df_primary)


In [ ]:
df_sorted = df_primary.sort_values(id_col)

lat_threshold = 0.0003592535
lon_threshold = 0.00064109755

max_lat_delta = (
    df_sorted.groupby("stop_id")["stop_lat"]
    .transform(lambda x: (x.mean() - x).abs().max())
)
max_lon_delta = (
    df_sorted.groupby("stop_id")["stop_lon"]
    .transform(lambda x: (x.mean() - x).abs().max())
)
df_sorted["stable_loc"] = (max_lat_delta < lat_threshold) & (
            max_lon_delta < lon_threshold)  #~40m (~56.6m in diagonal movement) at 56N
stable_means = (
    df_sorted.loc[df_sorted["stable_loc"]]
    .groupby("stop_id", as_index=True)[["stop_lat", "stop_lon"]]
    .mean()
)
stable_means
# write back means only for stable rows
stable_mask = df_sorted["stable_loc"]
stable_mask
df_sorted.loc[stable_mask, "stop_id"]
stable_means["stop_lat"]
df_sorted.loc[stable_mask, "stop_id"].map(stable_means["stop_lat"])
df_sorted.loc[stable_mask, "stop_lat"] = df_sorted.loc[stable_mask, "stop_id"].map(stable_means["stop_lat"])
df_sorted.loc[stable_mask, "stop_lon"] = df_sorted.loc[stable_mask, "stop_id"].map(stable_means["stop_lon"])
if df_sorted['stable_loc'].any():
    print(
        "Warning: stop_id with max delta lat/lon above 40m threshold:\n",
        df_sorted.loc[df_sorted['stable_loc'], ["stop_id"]].drop_duplicates(),
        "\nDuplicated stop_id with lat/lon differences > 40, will get new stop_id."
    )
# warning flags
high_lat_delta = (max_lat_delta >= lat_threshold)
high_lon_delta = (max_lon_delta >= lon_threshold)
high_lon_delta
if high_lat_delta.any() or high_lon_delta.any():
    print(
        "Warning: stop_id with max delta lat/lon above 40m threshold:\n",
        df_sorted.loc[high_lat_delta | high_lon_delta, ["stop_id"]].drop_duplicates(),
        "\nDuplicated stop_id with lat/lon differences > 40, will get new stop_id."
    )
#drop duplicates with same stop_id/_lat/_lon. lat/lon has been average by stop_id if within 40m. Duplicated stop_id with delta lat/lon, will get new stop_id below.
df_sorted = df_sorted.drop_duplicates(subset=["stop_id", "stop_lat", "stop_lon"], inplace=False)
final_count = len(df_sorted)
duplicates_removed = initial_count - final_count
duplicates_removed